# Assignment pages — HWT English on ruled paper

This notebook is the **neural English** path. It needs a style PNG of your writing.

For the **CPU Caveat + math** path used for CS743 Assignment 2 (no GPU, full solutions PDF), do **not** use this notebook. On the PC run:

`PYTHONPATH=src python src/textwritter/assignment_engine.py samples/assignment2_solutions.txt output/a2/Assignment2_solutions.pdf --style caveat --ink blue`

See `docs/ASSIGNMENT_PAGES.md` and `AGENTS.md`.

This HWT notebook:

- Paper looks like the 35-page lab book (red margin, blue lines).
- English lines go through **HWT** (use this for many pages; One-DM is too slow).
- Lines that are mostly math/symbols fall back to a handwriting **font** so the page is not empty.
- This will **not** reproduce formulas, arrows, or your exact letter shapes from a scan.

In [ ]:
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content/.config")
REPO_URL = "https://github.com/Jeevant010/Text_Writter"
if IN_COLAB:
    root = pathlib.Path("/content/Text_Writter")
    if not (root / "src/textwritter/write_assignment.py").exists():
        !git clone --depth 1 -b new-setup {REPO_URL} {root}
    os.chdir(root)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    os.chdir(root)
print("repo root:", pathlib.Path.cwd())

In [ ]:
missing = []
for mod in ["torch", "cv2", "PIL", "numpy", "gdown"]:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
if missing:
    !{sys.executable} -m pip install -q -r requirements.txt

## Style image

Upload **one page of English writing** (Lab 1 page 1 is better than a tensor-math page).
On your PC you can also run:
`python src/textwritter/extract_style.py --pdf Ui23cs30Lab1.pdf --page 1`

In [ ]:
STYLE = None
pathlib.Path("samples").mkdir(exist_ok=True)
if IN_COLAB:
    from google.colab import files
    print("Upload ONE PNG/JPG of your handwriting (a full English page)")
    up = files.upload()
    if up:
        name = list(up)[0]
        dest = pathlib.Path("samples") / name
        pathlib.Path(name).replace(dest)
        STYLE = str(dest)
else:
    import glob
    cands = sorted(glob.glob("samples/my_handwriting*.png") + glob.glob("samples/my_handwriting*.jpg")
                   + glob.glob("samples/*.png") + glob.glob("samples/*.jpg"))
    cands = [p for p in cands if "sample_hello" not in p]
    STYLE = cands[0] if cands else None
print("style:", STYLE)

## Your new assignment text (English)

Paste what you want **written**. Keep math as words (`tensor order is 2`, `shape 4 by 6`) if you want it in the neural style. Raw `∈ ℝ` lines will use the font fallback.

In [ ]:
ASSIGNMENT = """
Assignment 1

Problem 1. State the tensor order, the shape, and the total number of entries.

A scalar has tensor order 0. Shape is empty. Total entries is 1.

A vector in R 7 has tensor order 1. Shape is 7. Total entries is 7.

A matrix 4 by 6 has tensor order 2. Total entries is 24.

Problem 2. Tensor order is the number of independent axes required to identify a single element.

Number of entries is the product of all shape dimensions.
""".strip()

ENGINE = "hwt"  # keep hwt for multi-page; onedm is for short tests only

In [ ]:
from textwritter.write_assignment import write_assignment
from IPython.display import Image as IImage, display

result = write_assignment(ASSIGNMENT, style=STYLE, engine=ENGINE)
print(result["pdf"])
for p in result["pages"][:4]:
    display(IImage(filename=str(p), width=720))